# Stop classification — Stage B: matching current night train stops to OSM stations

See `STOP_CLASSIFICATION.md` Stage B. Output of this notebook: one confirmed OSM match per current night train stop, plus review reports for anything the pipeline can't decide on its own (never a silent drop, per the design doc's keep-it-in / auditable principles).

In [1]:
import re

import geopandas as gpd
import numpy as np
import pandas as pd
from rapidfuzz import fuzz
from unidecode import unidecode

## 1. Load & clean

The OSM export contains one stray row where every column literally equals its own header (a header line that leaked into the data, likely from a concatenated export) — drop it explicitly rather than let it silently corrupt string comparisons downstream.

In [2]:
osm = pd.read_csv("data/bahnhoefe_stops_sorted.csv")
osm = osm[osm["ID"] != "ID"]  # drop embedded duplicate header row(s)

schedule = pd.read_csv("data/B-o-T_DataBase_stop_times.csv", decimal=",")
schedule = schedule[["train_stop_id", "trip_id", "stop_sequence", "stop_id", "stop_lat", "stop_lon"]]

osm["Latitude"] = pd.to_numeric(osm["Latitude"], errors="coerce")
osm["Longitude"] = pd.to_numeric(osm["Longitude"], errors="coerce")
schedule["stop_lat"] = pd.to_numeric(schedule["stop_lat"], errors="coerce")
schedule["stop_lon"] = pd.to_numeric(schedule["stop_lon"], errors="coerce")

In [3]:
# A handful of schedule rows have no coordinates at all (arrival/departure-only marker rows).
# They can't be geo-matched, so drop them here rather than let them silently fail the join later.
rows_before = len(schedule)
schedule = schedule.dropna(subset=["stop_lat", "stop_lon"])
print(f"Dropped {rows_before - len(schedule)} schedule rows with missing coordinates")

Dropped 197 schedule rows with missing coordinates


## 2. Collapse the schedule to one row per stop

`schedule` has one row per (trip, stop) — the same physical stop appears once per train that calls there, and duplicate reports of the same stop don't always agree on coordinates (copy/paste errors in the source data, e.g. one `Bern` entry carrying Amsterdam's coordinates).

We resolve this with a **medoid**: for each `stop_id`, pick the reported coordinate that is closest (in total distance) to all other reports of that same stop. That's robust to a single bad outlier report, unlike a plain mean/first-row pick. Spread between reports is also recorded, so stops where reports disagree by more than a sane GPS-jitter margin are flagged for manual review instead of silently resolved — a two-report split (like the Bern/Amsterdam case) is exactly the situation the medoid can't resolve reliably from the numbers alone.

In [4]:
COORD_CONFLICT_THRESHOLD_M = 100  # above this spread, treat as a genuine data conflict, not GPS jitter


def resolve_stop_coords(group: pd.DataFrame) -> pd.Series:
    """Pick one representative coordinate per stop_id via medoid, and flag disagreement."""
    lat = group["stop_lat"].to_numpy()
    lon = group["stop_lon"].to_numpy()

    if len(lat) == 1:
        return pd.Series({"stop_lat": lat[0], "stop_lon": lon[0], "n_reports": 1, "coord_spread_m": 0.0})

    # Equirectangular approximation is accurate enough at station-cluster scale (a few km at most)
    lat_m = lat * 111_320
    lon_m = lon * 111_320 * np.cos(np.radians(lat.mean()))
    pairwise_dist = np.sqrt((lat_m[:, None] - lat_m[None, :]) ** 2 + (lon_m[:, None] - lon_m[None, :]) ** 2)

    medoid_idx = pairwise_dist.sum(axis=1).argmin()
    return pd.Series(
        {
            "stop_lat": lat[medoid_idx],
            "stop_lon": lon[medoid_idx],
            "n_reports": len(lat),
            "coord_spread_m": pairwise_dist.max(),
        }
    )


stops = schedule.groupby("stop_id").apply(resolve_stop_coords).reset_index()
print(f"Schedule collapsed from {len(schedule)} rows to {len(stops)} unique stops")

Schedule collapsed from 3672 rows to 601 unique stops


In [5]:
coord_conflicts = stops[stops["coord_spread_m"] > COORD_CONFLICT_THRESHOLD_M].sort_values(
    "coord_spread_m", ascending=False
)
coord_conflicts.to_csv("data/schedule_coord_conflicts_report.csv", index=False)
print(f"{len(coord_conflicts)} stops have disagreeing coordinate reports — see schedule_coord_conflicts_report.csv")
coord_conflicts

10 stops have disagreeing coordinate reports — see schedule_coord_conflicts_report.csv


,stop_id,stop_lat,stop_lon,n_reports,coord_spread_m
541,Valladolid-Campo Grande,41.64305,-4.726651,3.0,775972.406722
282,Lausanne,46.51708,6.629193,6.0,665402.809247
170,Fribourg / Freiburg,46.80321,7.151174,4.0,642798.330234
50,Bern,46.94839,7.436390,6.0,633545.823521
585,Zürich HB,47.37818,8.540212,24.0,620178.372387
379,Olten,47.34996,7.903703,4.0,601922.947366
26,Baden,47.47656,8.307640,2.0,596039.770242
12,Antwerpen Centraal,51.21728,4.421142,12.0,133573.688673
440,Rotterdam Centraal,51.92438,4.469746,12.0,58615.180006
457,Schipol Airport,52.30959,4.762805,9.0,12148.874415


## 3. Normalize names before comparing them

`Name (ASCII)` on the OSM side is already transliterated, but the schedule's `stop_id` (which holds the station *name*, not an id) is not — e.g. `București Nord`, `Timişoara Nord`. Comparing an accented name against a transliterated one drags the fuzzy match score down for no real reason. Normalize both sides the same way — strip accents, lowercase, collapse punctuation — before scoring.

In [6]:
def normalize_name(name: str) -> str:
    ascii_name = unidecode(str(name)).lower()
    return re.sub(r"[^a-z0-9]+", " ", ascii_name).strip()


osm["name_norm"] = osm["Name (ASCII)"].map(normalize_name)
stops["name_norm"] = stops["stop_id"].map(normalize_name)

## 4. Geo-match: OSM stations vs. schedule stops

Two fixes here vs. the previous approach:

- **Projection.** `EPSG:25832` is a UTM zone centered on Germany — fine for DE, but it distorts   badly for anything far from that zone (this network spans from Portugal to Ukraine).   `EPSG:3035` (ETRS89-LAEA Europe) is the standard pan-European equal-area CRS and keeps meter-based   distances accurate continent-wide, so it's the right choice for a Europe-scale nearest-neighbour join.
- **Nearest-neighbour join.** `sjoin_nearest` with a `max_distance` replaces the buffer+`contains`   join — it directly returns the true point-to-point distance (no need to re-look-up geometries   afterwards) and naturally gives one nearest candidate per schedule stop instead of every OSM   station within a fixed radius.

In [7]:
MAX_MATCH_RADIUS_M = 1_500  # widen-search ceiling from STOP_CLASSIFICATION.md Stage B (start 500 m, widen to 1.5 km)

osm_gdf = gpd.GeoDataFrame(
    osm, geometry=gpd.points_from_xy(osm.Longitude, osm.Latitude), crs="EPSG:4326"
).to_crs("EPSG:3035")

stops_gdf = gpd.GeoDataFrame(
    stops, geometry=gpd.points_from_xy(stops.stop_lon, stops.stop_lat), crs="EPSG:4326"
).to_crs("EPSG:3035")

candidates = gpd.sjoin_nearest(
    stops_gdf, osm_gdf, max_distance=MAX_MATCH_RADIUS_M, distance_col="distance_m", how="left"
)
print(f"{candidates['index_right'].isna().sum()} schedule stops have no OSM candidate within {MAX_MATCH_RADIUS_M} m")

31 schedule stops have no OSM candidate within 1500 m


In [8]:
candidates["name_score"] = candidates.apply(
    lambda r: fuzz.token_sort_ratio(r["name_norm_left"], r["name_norm_right"])
    if pd.notna(r["index_right"])
    else np.nan,
    axis=1,
)

# Both terms are scaled 0-100 so the weights are directly comparable regardless of MAX_MATCH_RADIUS_M
candidates["score"] = 0.7 * candidates["name_score"] + 0.3 * (
    100 - candidates["distance_m"] / (MAX_MATCH_RADIUS_M / 100)
)

## 5. Resolve to one match per schedule stop

`stops` is already unique per `stop_id`, so `sjoin_nearest` already returns at most one OSM candidate per schedule stop (ties broken arbitrarily by geopandas) — no separate dedup step against `index_right` is needed here, and it would only reintroduce the earlier bug of two conflicting `drop_duplicates` passes silently overwriting each other's filtering.

What *can* still happen is the reverse: two differently-named schedule stops both resolve to the **same** OSM station (e.g. `Wien Hbf` and `Wien Hbf (Autoreisezuganlage)`, or genuine duplicate spellings like `Göteborg C` / `Göteborg Centralen station`). That's a real signal worth a look, not something to hide by silently keeping only the higher-scoring one.

In [9]:
matched = candidates[candidates["index_right"].notna()].copy()
unmatched = candidates[candidates["index_right"].isna()].copy()

unmatched[["stop_id", "stop_lat", "stop_lon", "n_reports", "coord_spread_m"]].to_csv(
    "data/unmatched_report.csv", index=False
)
print(f"{len(unmatched)} schedule stops written to unmatched_report.csv (no OSM match within {MAX_MATCH_RADIUS_M} m)")

31 schedule stops written to unmatched_report.csv (no OSM match within 1500 m)


In [10]:
same_osm_station = matched[matched.duplicated("ID", keep=False)].sort_values("ID")
same_osm_station.to_csv("data/duplicate_osm_matches_report.csv", index=False)
print(f"{same_osm_station['ID'].nunique()} OSM stations were matched by more than one schedule stop name")
print("-> see duplicate_osm_matches_report.csv (likely alternate spellings of the same stop, worth consolidating upstream)")

10 OSM stations were matched by more than one schedule stop name
-> see duplicate_osm_matches_report.csv (likely alternate spellings of the same stop, worth consolidating upstream)


## 6. Confidence labelling

Per Stage B: record a confidence label per match instead of a single hard cutoff, so low-confidence matches stay in the output (keep-it-in principle) but are clearly marked for the QGIS review pass.

In [11]:
def confidence_label(row: pd.Series) -> str:
    if row["distance_m"] <= 500 and row["name_score"] >= 85:
        return "exact"
    if row["distance_m"] <= 500 and row["name_score"] >= 60:
        return "geo_name"
    if row["distance_m"] <= 500:
        return "geo_only"
    return "ambiguous"


matched["match_confidence"] = matched.apply(confidence_label, axis=1)
matched["match_confidence"].value_counts()

match_confidence
exact        371
geo_name     104
geo_only      71
ambiguous     24
Name: count, dtype: int64

## 7. Final output

In [12]:
result = matched[
    [
        "stop_id",
        "stop_lat",
        "stop_lon",
        "ID",
        "Name",
        "Name (ASCII)",
        "Länderkürzel",
        "Zeitzone",
        "distance_m",
        "name_score",
        "score",
        "match_confidence",
        "n_reports",
        "coord_spread_m",
    ]
].rename(columns={"ID": "osm_id", "Name": "osm_name", "Länderkürzel": "osm_country", "Zeitzone": "osm_timezone"})

result.to_csv("data/matched_stops.csv", index=False)
result

,stop_id,stop_lat,stop_lon,osm_id,osm_name,Name (ASCII),osm_country,osm_timezone,distance_m,name_score,score,match_confidence,n_reports,coord_spread_m
0,Aachen Hbf,50.76794,6.091263,3070631211,Aachen Hauptbahnhof,Aachen Central Train Station,DE,CET,21.063177,36.842105,55.368210,geo_only,12.0,0.223186
1,Aarau,47.39043,8.045701,3080746008,Aarau,Aarau,CH,CET,438.501391,100.000000,91.229972,exact,2.0,0.000000
2,Aberdeen,57.14469,-2.096399,7149963979,Aberdeen,Aberdeen,GB,GMT,239.945496,100.000000,95.201090,exact,2.0,0.000000
3,Afyon A Çetinkaya,38.76423,30.552503,142218952,Ali Çetinkaya Garı,Ali Cetinkaya Gari,TR,TRT,49.879425,62.857143,73.002412,geo_name,2.0,0.000000
4,Alba Iulia,46.05756,23.576502,260396172,Alba Iulia,Alba Iulia,RO,EET,149.253489,100.000000,97.014930,exact,2.0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
596,İzmir Çiğli,38.51831,27.047384,4556769264,Egekent,Egekent,TR,TRT,1222.067729,11.111111,13.336423,ambiguous,4.0,0.000000
597,İzmit,40.76551,29.939991,11082593543,Kırkikievler,Kirkikievler,TR,TRT,597.469364,23.529412,34.521201,ambiguous,1.0,0.000000
598,Łeba,54.75691,17.553198,563273123,Łeba,Leba,PL,CET,105.909858,100.000000,97.881803,exact,2.0,0.000000
599,Łódź Kaliska,51.75791,19.430861,1974456578,Łódź Kaliska,Lodz Kaliska,PL,CET,24.077786,100.000000,99.518444,exact,4.0,0.000000


In [13]:
print("OSM rows:", len(osm))
print("Schedule rows (raw):", rows_before)
print("Schedule stops (unique):", len(stops))
print("Matched:", len(result))
print("Unmatched:", len(unmatched))
print("Coordinate conflicts flagged:", len(coord_conflicts))
print("OSM stations claimed by >1 schedule stop:", same_osm_station["ID"].nunique())
result["match_confidence"].value_counts()

OSM rows: 48616
Schedule rows (raw): 3869
Schedule stops (unique): 601
Matched: 570
Unmatched: 31
Coordinate conflicts flagged: 10
OSM stations claimed by >1 schedule stop: 10


match_confidence
exact        371
geo_name     104
geo_only      71
ambiguous     24
Name: count, dtype: int64